# NeuralRNN Built-in Cognitive Tasks Gallery

This notebook visualizes every built-in cognitive task in `neuralrnn.data.tasks`.
For each task we show:
- **Task goal**: what the network is asked to do.
- **Input/output dimensions**: tensor shapes and channel semantics.
- **Trial structure**: stimulus, delay, decision, and loss epochs.
- **Representative trials**: line plots of inputs and targets, with shaded regions indicating where the loss mask is active.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from pathlib import Path
from neuralrnn.data import load_dataset

plt.rcParams['figure.dpi'] = 100

# All figures are saved under FIG_DIR (this notebook trains no models).
FIG_DIR = Path("./figs/cognitive_tasks")
FIG_DIR.mkdir(parents=True, exist_ok=True)


def plot_task_io(ds, title, input_labels=None, output_labels=None, n_show=2,
                 save_prefix=None):
    """Plot inputs and targets as line traces for a few trials.

    Each trial is shown in its own figure. Inputs are stacked vertically
    (one row per input channel) on the left; outputs are stacked vertically
    (one row per output channel) on the right. Loss-mask regions are
    highlighted by a light-grey background and by a red bar at the top of
    each output panel.

    If `save_prefix` is given, each trial figure is saved to
    ``FIG_DIR / f"{save_prefix}_trial{i+1}.png"``.
    """
    n_show = min(n_show, ds.inputs.shape[0])
    n_in = ds.inputs.shape[-1]
    n_out = ds.targets.shape[-1]

    if input_labels is None:
        input_labels = [f'input {i}' for i in range(n_in)]
    if output_labels is None:
        output_labels = [f'output {i}' for i in range(n_out)]

    t = np.arange(ds.inputs.shape[1])
    figures = []

    for i in range(n_show):
        fig = plt.figure(figsize=(14, 1.8 * max(n_in, n_out)))
        outer = GridSpec(1, 2, figure=fig, wspace=0.3)
        left = outer[0, 0].subgridspec(n_in, 1, hspace=0.15)
        right = outer[0, 1].subgridspec(n_out, 1, hspace=0.15)

        # ---- inputs ----
        for ch in range(n_in):
            ax = fig.add_subplot(left[ch, 0])
            ax.plot(t, ds.inputs[i, :, ch], lw=1.2, color=f'C{ch % 10}')
            ax.set_ylabel(input_labels[ch], fontsize=9)
            ax.set_xlim(t[0], t[-1])
            ax.autoscale(enable=True, axis='y', tight=True)
            if ch == 0:
                ax.set_title('Inputs')
            plt.setp(ax.get_xticklabels(), visible=(ch == n_in - 1))

        # ---- outputs ----
        for ch in range(n_out):
            ax = fig.add_subplot(right[ch, 0], sharex=fig.axes[0])
            ax.plot(t, ds.targets[i, :, ch], lw=1.5, color='C0')

            mask = ds.mask[i, :, ch].numpy() if hasattr(ds.mask, 'numpy') else np.asarray(ds.mask[i, :, ch])
            mask = mask.astype(float)
            if mask.ndim == 0:
                mask = np.full_like(t, float(mask), dtype=float)

            y_min, y_max = ax.get_ylim()
            ax.fill_between(t, y_min, y_max, where=mask > 0, color='grey', alpha=0.12, interpolate=True)

            # Red bar indicating mask=1.
            bar_top = y_max + 0.05 * (y_max - y_min)
            bar_bottom = y_max + 0.01 * (y_max - y_min)
            ax.fill_between(t, bar_bottom, bar_top, where=mask > 0, color='C3', alpha=0.8, interpolate=True)
            ax.set_ylim(top=bar_top + 0.04 * (y_max - y_min))

            ax.set_ylabel(output_labels[ch], fontsize=9)
            if ch == 0:
                ax.set_title('Targets (grey shade / red bar = loss mask)')
            plt.setp(ax.get_xticklabels(), visible=(ch == n_out - 1))
            if ch == n_out - 1:
                ax.set_xlabel('Time step')

        fig.suptitle(f'{title} — trial {i+1}', fontsize=11, y=0.995)
        if save_prefix is not None:
            fig.savefig(FIG_DIR / f"{save_prefix}_trial{i+1}.png", dpi=150)
        figures.append(fig)

    return figures

## 1. Mante / Siegel-Miller: context-dependent decision making

**Paper**: Mante et al. (2013), *Nature*; also called Siegel-Miller task (Siegel et al., 2015).

**Goal**: A context cue tells the network to attend to either motion or color evidence; report the sign of the attended coherence.

**Input** (6 channels): `[motion_ctx, color_ctx, motion_r, motion_l, color_r, color_l]`.
**Target** (2 channels): `[choice_r, choice_l]` active during the decision epoch.

**Parameters**: `cohs` now customizable; default is `np.linspace(-0.2, 0.2, 6)`.

`siegel_miller` is kept as a backward-compatible alias for `mante`.

In [ ]:
ds_mante = load_dataset('mante', n_trials=2, n_coh=3, batch_size=4)
print('Conditions (first trial):', ds_mante.conditions[0])
plot_task_io(
    ds_mante,
    'Mante / Siegel-Miller: context-dependent decision making',
    input_labels=['motion_ctx', 'color_ctx', 'motion_r', 'motion_l', 'color_r', 'color_l'],
    output_labels=['choice_r', 'choice_l'],
    save_prefix='mante',
);

In [ ]:
ds_rdm = load_dataset('rdm', n_trials=4, batch_size=4)
print('First condition:', ds_rdm.conditions[0])
plot_task_io(
    ds_rdm,
    'RDM: integrate noisy coherence',
    input_labels=['coherence + noise'],
    output_labels=['choice sign'],
    save_prefix='rdm',
);

## 2. RDM: random-dot-motion perceptual decision making

**Paper**: Dubreuil et al. (2022), *Nature Neuroscience*.

**Goal**: Integrate noisy motion evidence and report its sign.

**Input** (1 channel): coherence signal + Gaussian noise.
**Target** (1 channel): `+1` / `-1` choice sign.

`two_afc` is kept as a backward-compatible alias for `rdm` (it represents the same motion-discrimination logic in a 2-output format).

In [ ]:
ds_dms_cont = load_dataset('dms_continuous', n_trials=2, n_coh=3, batch_size=4)
print('First condition:', ds_dms_cont.conditions[0])
plot_task_io(
    ds_dms_cont,
    'DMS continuous: match same-sign coherences',
    input_labels=['test_r', 'test_l', 'sample_r', 'sample_l'],
    output_labels=['match', 'non-match'],
    save_prefix='dms_continuous',
);

## 3. DMS continuous: delayed match-to-sample with continuous coherences

**Goal**: Two sequential stimuli are presented; report whether they have the same sign.

**Input** (4 channels): `[test_r, test_l, sample_r, sample_l]`.
**Target** (2 channels): `[match, non-match]`.

The mask is 1 during the pre-sample and decision epochs and 0 during the delay.

In [ ]:
ds_dms = load_dataset('dms', n_trials=2, batch_size=4)
print('First condition:', ds_dms.conditions[0])
plot_task_io(
    ds_dms,
    'DMS discrete: A/B match-to-sample',
    input_labels=['stim A', 'stim B'],
    output_labels=['match (+1) / non-match (-1)'],
    save_prefix='dms',
);

## 4. DMS discrete: A/B match-to-sample

**Paper**: Dubreuil et al. (2022), *Nature Neuroscience*.

**Goal**: Two one-hot stimuli (A/B) are presented; report match (+1) or non-match (-1).

**Input** (2 channels): one-hot A/B.
**Target** (1 channel): `+1` match / `-1` non-match.

In [ ]:
ds_wm_angle = load_dataset('wm_angle', n_trials=4, batch_size=4)
print('First condition:', ds_wm_angle.conditions[0])
plot_task_io(
    ds_wm_angle,
    'WM angle: remember and reproduce a circular angle',
    input_labels=['cos(theta)', 'sin(theta)'],
    output_labels=['cos(theta)', 'sin(theta)'],
    save_prefix='wm_angle',
);

## 5. WM angle: parametric working memory on a circular angle

**Goal**: Remember a continuous angle over a delay and reproduce it.

**Input / Target** (2 channels): `[cos(theta), sin(theta)]`.

The mask is 1 during stimulus and decision and 0 during the delay.

`parametric_wm` is kept as a backward-compatible alias.

In [ ]:
ds_wm_freq = load_dataset('wm_frequency', n_trials=4, batch_size=4)
print('First condition:', ds_wm_freq.conditions[0])
plot_task_io(
    ds_wm_freq,
    'WM frequency: frequency-comparison working memory',
    input_labels=['normalized freq'],
    output_labels=['(f1-f2)/span'],
    save_prefix='wm_frequency',
);

## 6. WM frequency: frequency-comparison working memory

**Paper**: Romo et al. (1999), *Nature*.

**Goal**: Compare two sequential vibrotactile frequencies after a variable delay.

**Input** (1 channel): normalized frequency.
**Target** (1 channel): signed comparison `(f1 - f2) / span`.

`romo` is kept as a backward-compatible alias.

In [ ]:
ds_raposo = load_dataset('raposo', n_trials=4, batch_size=4)
print('First condition:', ds_raposo.conditions[0])
plot_task_io(
    ds_raposo,
    'Raposo: multisensory decision making',
    input_labels=['visual', 'auditory', 'visual_ctx', 'auditory_ctx'],
    output_labels=['choice sign'],
    save_prefix='raposo',
);

## 7. Raposo: multisensory decision making

**Paper**: Raposo et al. (2014), *Nature Neuroscience*.

**Goal**: Attend to visual, auditory, or both modalities and report the dominant direction.

**Input** (4 channels): `[visual_stim, auditory_stim, visual_ctx, auditory_ctx]`.
**Target** (1 channel): choice sign.

## 8. Low-rank Mante: context-dependent DM (low-rank format)

**Paper**: Dubreuil et al. (2022), *Nature Neuroscience*; Valente et al. (2022), *NeurIPS*.

**Goal**: Same cognitive task as Mante, but using the low-rank RNN convention.

**Input** (4 channels): `[color_stim, motion_stim, color_ctx, motion_ctx]`.
**Target** (1 channel): scalar `+1` / `-1` decision sign.

This format is mechanically distinct from the 6-input / 2-output `mante` task above.

In [ ]:
ds_mante2 = load_dataset('mante2', n_trials=4, batch_size=4)
print('First condition:', ds_mante2.conditions[0])
plot_task_io(
    ds_mante2,
    'Mante2: context-dependent DM (low-rank RNN convention)',
    input_labels=['color', 'motion', 'color_ctx', 'motion_ctx'],
    output_labels=['choice sign'],
    save_prefix='mante2',
);

## 9. Neurogym tasks: unified interface and raw env access

[Neurogym](https://github.com/neurogym/neurogym) is an external library of cognitive-task
environments (optional dependency: `pip install -e '.[neurogym]'`). Any task registered in the
installed neurogym loads through the same `load_dataset` entry point — pass the env id
(case-insensitive, `-v0` optional); `neuralrnn.data.list_neurogym_datasets()` lists what is
available. Two registered shortcuts are `perceptual_decision_making` and `delay_comparison`.

With `n_trials=...`, a neurogym dataset pre-generates complete trials and exposes **the same
interface as the built-in tasks above** (`inputs` / `targets` / `mask` / `conditions`,
`get_all_trials()`, `len(ds)`): shorter trials are zero-padded, `mask` marks valid steps, and each
condition dict carries the native trial info plus `epochs` (period bounds) and `n_steps`. Without
`n_trials`, the dataset streams fixed-length windows for training (used by notebooks 01/04/05/...).

For advanced use (RL-style interaction, timing modification, neurogym wrappers), the raw
environment remains available as `ds.env` — an unwrapped `TrialEnv` supporting `new_trial()`,
`ob`, `gt`, `trial`, `dt`, `start_ind` / `end_ind`.

In [ ]:
# Any neurogym env id loads through load_dataset (dynamic passthrough)
from neuralrnn.data import list_neurogym_datasets
print(f'{len(list_neurogym_datasets())} neurogym envs available, e.g.:',
      list_neurogym_datasets()[:5])

# Trial-aligned mode: same interface as the built-in cognitive tasks
ds_ng = load_dataset('PerceptualDecisionMaking-v0', n_trials=4, batch_size=4, seed=0)
print('inputs:', tuple(ds_ng.inputs.shape), '| targets:', tuple(ds_ng.targets.shape),
      '| mask:', tuple(ds_ng.mask.shape), '| len:', len(ds_ng))
print('condition keys:', sorted(ds_ng.conditions[0]))
print('epochs of trial 0:', ds_ng.conditions[0]['epochs'])

# Raw env access is still available for advanced use (RL-style stepping, timing edits)
env = ds_ng.env
env.new_trial()
print('\nraw env: ob', env.ob.shape, '| gt', env.gt.shape, '| dt =', env.dt,
      '| trial:', env.trial)
# On-demand example trials from ANY neurogym dataset — including streaming
# ones — without creating a second dataset: ds.sample_trials(n)
trials = ds_ng.sample_trials(2, seed=0)
print('\nsample_trials:', tuple(trials.inputs.shape),
      '| targets:', tuple(trials.targets.shape), '| trials:', len(trials))


## Summary

| Task | Family | Input dim | Output dim | Key feature |
|---|---|---|---|---|
| `mante` / `siegel_miller` | Context-dependent DM | 6 | 2 | Context selects motion vs. color |
| `rdm` / `two_afc` | Perceptual DM | 1 | 1 | Noisy coherence integration |
| `dms_continuous` | Match-to-sample | 4 | 2 | Continuous coherences |
| `dms` | Match-to-sample | 2 | 1 | Discrete A/B symbols |
| `wm_angle` / `parametric_wm` | Parametric WM | 2 | 2 | Circular angle |
| `wm_frequency` / `romo` | Parametric WM | 1 | 1 | Frequency comparison |
| `raposo` | Multisensory DM | 4 | 1 | Visual / auditory / both |
| `mante2` | Context-dependent DM | 4 | 1 | Low-rank RNN convention |